In [ ]:
import pandas as pd
import numpy as np
import pickle 
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('data/estudantes_limpo.csv')

print(f"Dataset carregado! Total de linhas: {df.shape[0]} | Total de colunas: {df.shape[1]}")

In [ ]:
total_alunos = len(df)

# 1. Calculando as Priores P(C)
contagem_situacao = df['situacao'].value_counts()
p_sucesso = contagem_situacao['Sucesso'] / total_alunos
p_abandono = contagem_situacao['Abandono'] / total_alunos

print("=== 1. PROBABILIDADES PRIORES ===")
print(f"P(Sucesso): {p_sucesso:.4f} ({p_sucesso*100:.2f}%)")
print(f"P(Abandono): {p_abandono:.4f} ({p_abandono*100:.2f}%)")
print("\n" + "="*40 + "\n")

# 2. Calculando a Verossimilhança P(X|C) para as variáveis categóricas
print("=== 2. MATRIZES DE VEROSSIMILHANÇA (PROPORÇÕES EXATAS) ===")

tabela_bolsista = pd.crosstab(df['bolsista'], df['situacao'], normalize='columns')
print("\n--- Proporção de Bolsistas por Situação ---")
print(tabela_bolsista)

tabela_turno = pd.crosstab(df['turno'], df['situacao'], normalize='columns')
print("\n--- Proporção de Turnos por Situação ---")
print(tabela_turno)

tabela_sexo = pd.crosstab(df['sexo'], df['situacao'], normalize='columns')
print("\n--- Proporção de Sexo por Situação ---")
print(tabela_sexo)

In [ ]:

df_ml = df.copy()

df_ml['bolsista_num'] = df_ml['bolsista'].map({'Sim': 1, 'Não': 0})
df_ml['turno_num'] = df_ml['turno'].map({'Diurno': 1, 'Noturno': 0})
df_ml['sexo_num'] = df_ml['sexo'].map({'Masculino': 1, 'Feminino': 0})


df_ml['alvo'] = df_ml['situacao'].map({'Abandono': 1, 'Sucesso': 0})

X = df_ml[['bolsista_num', 'turno_num', 'sexo_num', 'idade_ingresso']]
y = df_ml['alvo']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dados divididos! Treino: {X_train.shape[0]} alunos | Teste: {X_test.shape[0]} alunos.")

In [ ]:

modelo_bayes = CategoricalNB()


modelo_bayes.fit(X_train, y_train)


y_pred_bayes = modelo_bayes.predict(X_test)


print("=== MÉTRICAS DO MODELO NAIVE BAYES ===")
print(f"Acurácia Geral: {accuracy_score(y_test, y_pred_bayes)*100:.2f}%\n")

print("--- Relatório de Classificação ---")
print(classification_report(y_test, y_pred_bayes, target_names=['Sucesso', 'Abandono']))

print("--- Matriz de Confusão ---")
print(confusion_matrix(y_test, y_pred_bayes))

In [11]:
# Salvando o modelo de Bayes dentro da pasta do projeto
with open('modelo_bayes.pkl', 'wb') as arquivo:
    pickle.dump(modelo_bayes, arquivo)

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
modelo_arvore = DecisionTreeClassifier(max_depth=5, random_state=42)

modelo_arvore.fit(X_train, y_train)

y_pred_arvore = modelo_arvore.predict(X_test)

print("=== MÉTRICAS DA ÁRVORE DE DECISÃO ===")
print(f"Acurácia Geral: {accuracy_score(y_test, y_pred_arvore)*100:.2f}%\n")
print(classification_report(y_test, y_pred_arvore, target_names=['Sucesso', 'Abandono']))

=== MÉTRICAS DA ÁRVORE DE DECISÃO ===
Acurácia Geral: 70.11%

              precision    recall  f1-score   support

     Sucesso       0.74      0.80      0.77       449
    Abandono       0.63      0.54      0.58       277

    accuracy                           0.70       726
   macro avg       0.68      0.67      0.67       726
weighted avg       0.70      0.70      0.70       726



In [ ]:
modelo_knn = KNeighborsClassifier(n_neighbors=5)

modelo_knn.fit(X_train, y_train)

y_pred_knn = modelo_knn.predict(X_test)

print("=== MÉTRICAS DO KNN ===")
print(f"Acurácia Geral: {accuracy_score(y_test, y_pred_knn)*100:.2f}%\n")
print(classification_report(y_test, y_pred_knn, target_names=['Sucesso', 'Abandono']))

=== MÉTRICAS DO KNN ===
Acurácia Geral: 67.49%

              precision    recall  f1-score   support

     Sucesso       0.75      0.72      0.73       449
    Abandono       0.57      0.61      0.59       277

    accuracy                           0.67       726
   macro avg       0.66      0.66      0.66       726
weighted avg       0.68      0.67      0.68       726



In [18]:
# Salvando a Árvore de Decisão
with open('modelo_arvore.pkl', 'wb') as arquivo:
    pickle.dump(modelo_arvore, arquivo)

# Salvando o KNN
with open('modelo_knn.pkl', 'wb') as arquivo:
    pickle.dump(modelo_knn, arquivo)